## Load the data!

In [ ]:
# Imports/installs

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import holidays
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configuration - 5 high-traffic zones with diverse demand patterns
# Zone IDs match TLC Taxi Zone lookup table
ZONES = {
    230: 'Times Sq/Theatre District',  # Manhattan - entertainment/dining
    261: 'World Trade Center',          # Manhattan - financial district
    79: 'East Village',                 # Manhattan - nightlife/restaurants
    237: 'Upper East Side South',       # Manhattan - residential/retail
    132: 'JFK Airport'                  # Queens - transportation hub
}

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
MONTHS = list(range(1, 13))
HOUR_RANGE = list(range(6, 23))  # business hours 6am-10pm
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

In [ ]:
# Load zone lookup table for enrichment
# This adds borough and full zone name information
def load_zone_lookup():
  """Load the TLC taxi zone lookup table"""
  zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
  try:
    zones_df = pd.read_csv(zone_url)
    # rename for clarity
    zones_df = zones_df.rename(columns={
      'LocationID': 'zone_id',
      'Borough': 'borough',
      'Zone': 'zone_full_name',
      'service_zone': 'service_zone'
    })
    return zones_df
  except:
    print("Could not load zone lookup table, will use zone IDs only")
    return None

In [ ]:
def download_parquet(year, month):
  filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
  url = BASE_URL + filename

  try:
    print(f"  Downloading {filename}...", end=" ")
    df = pd.read_parquet(url)
    print(f"Done - {len(df):,} rows")
    return df
  except Exception as e:
    print(f"Failed: {e}")
    return None

In [ ]:
def filter_and_aggregate(df, year, month):
  if df is None or len(df) == 0:
    return None

  # check if required columns exist (column names are standardized in parquet files)
  required_cols = ['tpep_pickup_datetime', 'PULocationID', 'passenger_count',
           'fare_amount', 'trip_distance']
  if not all(col in df.columns for col in required_cols):
    print(f"    Missing columns in {year}-{month:02d}, skipping")
    return None

  # filter to our 5 zones
  df = df[df['PULocationID'].isin(ZONES.keys())].copy()

  if len(df) == 0:
    print(f"    No data for target zones in {year}-{month:02d}")
    return None

  # get hour and filter to business hours
  df['hour'] = pd.to_datetime(df['tpep_pickup_datetime']).dt.hour
  df = df[df['hour'].isin(HOUR_RANGE)]

  if len(df) == 0:
    return None

  # floor to hourly timestamp
  df['timestamp'] = pd.to_datetime(df['tpep_pickup_datetime']).dt.floor('H')

  # aggregate to hourly counts by zone
  hourly = df.groupby(['PULocationID', 'timestamp']).agg({
    'passenger_count': 'sum',
    'fare_amount': 'sum',
    'trip_distance': 'mean',
    'VendorID': 'count'
  }).reset_index()

  hourly.columns = ['zone_id', 'timestamp', 'customers', 'revenue',
            'avg_trip_distance', 'trip_count']

  hourly['avg_fare'] = hourly['revenue'] / hourly['trip_count']

  print(f"Aggregated to {len(hourly)} hourly records")

  return hourly


In [ ]:
def load_all_taxi_data():
  print("Starting data download...")

  # load zone lookup table
  zone_lookup = load_zone_lookup()

  all_data = []
  total_months = len(YEARS) * len(MONTHS)
  processed = 0
  errors = 0

  for year in YEARS:
    print(f"Processing {year}:")

    for month in MONTHS:
      processed += 1

      df_raw = download_parquet(year, month)

      if df_raw is not None:
        df_hourly = filter_and_aggregate(df_raw, year, month)

        if df_hourly is not None:
          all_data.append(df_hourly)
        else:
          errors += 1
      else:
        errors += 1

      del df_raw

  print(f"\nDownload complete!")
  print(f"Successfully processed: {processed - errors}/{total_months} months")
  print(f"Errors or missing data: {errors}/{total_months} months")

  if len(all_data) == 0:
    print("ERROR: No data was loaded")
    return None

  print("\nCombining all data...")
  final_df = pd.concat(all_data, ignore_index=True)
  final_df = final_df.sort_values(['zone_id', 'timestamp']).reset_index(drop=True)

  # add zone information from lookup table
  if zone_lookup is not None:
    print("Merging zone information...")
    final_df = final_df.merge(
      zone_lookup[['zone_id', 'borough', 'zone_full_name']],
      on='zone_id',
      how='left'
    )
  else:
    # fallback to basic zone names
    final_df['zone_name'] = final_df['zone_id'].map(ZONES)

  print("\n" + "="*70)
  print("FINAL DATASET:")
  print("="*70)
  print(f"Total rows: {len(final_df):,}")
  print(f"Total columns: {len(final_df.columns)}")
  print(f"Date range: {final_df['timestamp'].min()} to {final_df['timestamp'].max()}")
  print(f"Memory usage: {final_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
  print("\nRows per zone:")
  if 'zone_full_name' in final_df.columns:
    print(final_df.groupby(['zone_id', 'zone_full_name']).size())
  else:
    print(final_df.groupby('zone_id').size())
  print("="*70)

  return final_df


In [ ]:
print("Starting data collection process...")
df = load_all_taxi_data()

if df is not None:
  # save to colab
  print("\nSaving to Colab filesystem...")
  df.to_csv('taxi_hourly_data.csv', index=False)
  print("Saved as: taxi_hourly_data.csv")

  # save to google drive
  try:
    drive_path = '/content/drive/MyDrive/nyc_taxi_hourly.csv'
    df.to_csv(drive_path, index=False)
    print(f"Also saved to Google Drive: {drive_path}")
  except:
    print("Could not save to Google Drive (mount it first if needed)")

  print("\nData loading complete!")
  print("To reload later: df = pd.read_csv('taxi_hourly_data.csv')")
else:
  print("Data loading failed - check errors above")


Starting data collection process...
Starting data download...
Processing 2019:

KeyboardInterrupt: 

In [ ]:
print("="*70)
print("COLLECTING WEATHER DATA")
print("="*70)

# Configuration
NYC_LAT = 40.7128
NYC_LON = -74.0060
START_DATE = '2019-01-01'
END_DATE = '2024-12-31'

WEATHER_FEATURES = [
    'temperature_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'snowfall',
    'weathercode',
    'cloudcover',
    'windspeed_10m',
    'relativehumidity_2m'
]

COLLECTING WEATHER DATA


In [ ]:
def fetch_openmeteo_weather(start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": NYC_LAT,
        "longitude": NYC_LON,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": WEATHER_FEATURES,
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "America/New_York"
    }

    print(f"Fetching weather data from {start_date} to {end_date}...")
    response = requests.get(url, params=params, timeout=30)

    if response.status_code != 200:
        raise Exception(f"API request failed: {response.status_code}")

    data = response.json()
    hourly = data['hourly']

    df = pd.DataFrame({
        'timestamp': pd.to_datetime(hourly['time']),
        'temperature': hourly['temperature_2m'],
        'apparent_temperature': hourly['apparent_temperature'],
        'precipitation': hourly['precipitation'],
        'rain': hourly['rain'],
        'snowfall': hourly['snowfall'],
        'weathercode': hourly['weathercode'],
        'cloudcover': hourly['cloudcover'],
        'windspeed': hourly['windspeed_10m'],
        'humidity': hourly['relativehumidity_2m']
    })

    print(f"Fetched {len(df):,} hourly records")
    return df


In [ ]:
def fetch_weather_in_chunks(start_date, end_date):
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    all_data = []
    current = start
    chunk_num = 1

    total_days = (end - start).days
    total_chunks = (total_days // 365) + 1

    print(f"Fetching data in {total_chunks} chunks\n")

    while current < end:
        chunk_end = min(current + timedelta(days=365), end)

        print(f"Chunk {chunk_num}/{total_chunks}:")
        df_chunk = fetch_openmeteo_weather(
            current.strftime('%Y-%m-%d'),
            chunk_end.strftime('%Y-%m-%d')
        )
        all_data.append(df_chunk)

        current = chunk_end + timedelta(days=1)
        chunk_num += 1
        print()

    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.sort_values('timestamp').reset_index(drop=True)
    return final_df


In [ ]:
def add_weather_features(df):
    # Binary indicators
    df['is_raining'] = (df['rain'] > 0).astype(int)
    df['is_snowing'] = (df['snowfall'] > 0).astype(int)
    df['temp_gap'] = df['temperature'] - df['apparent_temperature']

    # Weather code interpretation
    def interpret_weather_code(code):
        if pd.isna(code):
            return 'Unknown'
        code = int(code)
        if code == 0:
            return 'Clear'
        elif code <= 3:
            return 'Partly Cloudy'
        elif code in [45, 48]:
            return 'Fog'
        elif 51 <= code <= 67:
            return 'Rain'
        elif 71 <= code <= 86:
            return 'Snow'
        elif code >= 95:
            return 'Thunderstorm'
        else:
            return 'Other'

    df['conditions'] = df['weathercode'].apply(interpret_weather_code)

    # Extreme weather
    df['is_extreme_cold'] = (df['apparent_temperature'] < 20).astype(int)
    df['is_extreme_heat'] = (df['apparent_temperature'] > 90).astype(int)
    df['is_windy'] = (df['windspeed'] > 20).astype(int)

    return df

weather_df = fetch_weather_in_chunks(START_DATE, END_DATE)

# Save to Drive
weather_path = '/content/drive/MyDrive/nyc_weather_hourly.csv'
weather_df.to_csv(weather_path, index=False)
print(f"\nSaved to: {weather_path}")


Fetching data in 7 chunks

Chunk 1/7:
Fetching weather data from 2019-01-01 to 2020-01-01...
Fetched 8,784 hourly records

Chunk 2/7:
Fetching weather data from 2020-01-02 to 2021-01-01...
Fetched 8,784 hourly records

Chunk 3/7:
Fetching weather data from 2021-01-02 to 2022-01-02...
Fetched 8,784 hourly records

Chunk 4/7:
Fetching weather data from 2022-01-03 to 2023-01-03...
Fetched 8,784 hourly records

Chunk 5/7:
Fetching weather data from 2023-01-04 to 2024-01-04...
Fetched 8,784 hourly records

Chunk 6/7:
Fetching weather data from 2024-01-05 to 2024-12-31...
Fetched 8,688 hourly records


Saved to: /content/drive/MyDrive/nyc_weather_hourly_2019_2024.csv
